## Clear the text

In [ ]:
import pandas as pd
import re
import html

def remove_emoji(text):
    emoji_pattern = re.compile(
        "[" 
        u"\U0001F600-\U0001F64F"
        u"\U0001F300-\U0001F5FF"
        u"\U0001F680-\U0001F6FF"
        u"\U0001F1E0-\U0001F1FF"
        "]+", flags=re.UNICODE)
    return emoji_pattern.sub(r'', text)

def clean_text(text: str) -> str:
    if pd.isna(text):
        return ""
    text = html.unescape(text)
    text = re.sub(r"<.*?>", " ", text)
    text = remove_emoji(text)
    # remove ALL quotes, double or single, anywhere in text
    text = text.replace('"', '').replace("'", '').replace('“','').replace('”','').replace('‘','').replace('’','').replace('`','')
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df = pd.read_csv("./filtered_dataset.csv")
df["clean_sentence"] = df["sentence"].apply(clean_text)
df.to_csv("cleaned_dataset.csv", index=False)


## Showing the data frequency

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load dataset (update path if necessary)
df = pd.read_csv("./cleaned_dataset.csv")

# --- Step 1: Keep only needed columns ---
df = df[['sentence', 'intensity_level']]

# --- Step 2: Drop duplicates (if same sentence appears multiple times) ---
df.drop_duplicates(subset='sentence', inplace=True)

# --- Step 3: Drop rows with missing values ---
df.dropna(subset=['sentence', 'intensity_level'], inplace=True)

# --- Step 4: Check data types ---
df['intensity_level'] = df['intensity_level'].astype(int)

# --- Step 5: Strip whitespace & normalize Bangla sentences ---
df['sentence'] = df['sentence'].str.strip()

# --- Step 6: Check class distribution ---
print(df['intensity_level'].value_counts())

# --- Step 7: Plot class distribution ---
plt.figure(figsize=(6,4))
sns.countplot(data=df, x='intensity_level', palette='Set2')
plt.title("Class Distribution of Intensity Levels")
plt.xlabel("Intensity Level")
plt.ylabel("Count")
plt.show()

# --- Step 8: Save cleaned dataset for further steps ---
df.to_csv("cleaned_dataset.csv", index=False, encoding="utf-8-sig")

## Balance the dataset

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from collections import Counter

# ==============================
# Step 1: Load the dataset after label handling
# ==============================
df = pd.read_csv("./cleaned_dataset.csv")

print("Original class distribution:")
print(df['intensity_level'].value_counts().sort_index())

# ==============================
# Step 2: Handle imbalance by undersampling
# ==============================
# Find minimum class size (minority class)
min_class_size = df['intensity_level'].value_counts().min()

# Undersample each class to match minority class size (NO WARNING ✅)
balanced_df = df.groupby('intensity_level', group_keys=False).sample(
    n=min_class_size, random_state=42
).reset_index(drop=True)

print("\nBalanced class distribution:")
print(balanced_df['intensity_level'].value_counts().sort_index())
balanced_df.to_csv("balanced_dfv2.csv", index=False, encoding="utf-8-sig")


## Split the dataset

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os

# ============================
# Load dataset
# ============================
df = pd.read_csv("./balanced_dfv2.csv")

# Optional: shuffle dataset
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# ============================
# Train / Temp split (70% train, 30% temp)
# ============================
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['intensity_level_normalized'])

# ============================
# Temp split into validation/test (15% each)
# ============================
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['intensity_level_normalized'])

# ============================
# Save CSVs
# ============================
train_df.to_csv("train.csv", index=False)
val_df.to_csv("validation.csv", index=False)
test_df.to_csv("test.csv", index=False)

print("✅ Train/Validation/Test split done!")
print(f"Train: {len(train_df)}, Validation: {len(val_df)}, Test: {len(test_df)}")
